In [2]:
import json

In [3]:
with open("../output/players.json", "r", encoding="utf-8") as f:
    raw = json.load(f)
print(len(raw))
player_info = raw[0]["data"]["response"]
print(player_info)

33
[{'player': {'id': 883, 'name': 'Lee Grant', 'firstname': 'Lee', 'lastname': 'Grant', 'age': 39, 'birth': {'date': '1983-01-27', 'place': 'Hemel Hempstead', 'country': 'England'}, 'nationality': 'England', 'height': '193 cm', 'weight': '83 kg', 'injured': False, 'photo': 'https://media.api-sports.io/football/players/883.png'}, 'statistics': [{'team': {'id': 33, 'name': 'Manchester United', 'logo': 'https://media.api-sports.io/football/teams/33.png'}, 'league': {'id': 39, 'name': 'Premier League', 'country': 'England', 'logo': 'https://media.api-sports.io/football/leagues/39.png', 'flag': 'https://media.api-sports.io/flags/gb-eng.svg', 'season': 2022}, 'games': {'appearences': None, 'lineups': None, 'minutes': None, 'number': None, 'position': 'Goalkeeper', 'rating': None, 'captain': False}, 'substitutes': {'in': None, 'out': None, 'bench': None}, 'shots': {'total': None, 'on': None}, 'goals': {'total': None, 'conceded': None, 'assists': None, 'saves': None}, 'passes': {'total': None

In [19]:
import random


def generate_height_weight(position):

    if position == "Goalkeeper":
        height = random.randint(185, 200)
        weight = random.randint(78, 95)

    elif position == "Defender":
        height = random.randint(178, 195)
        weight = random.randint(72, 90)

    elif position == "Midfielder":
        height = random.randint(168, 188)
        weight = random.randint(65, 82)

    elif position == "Attacker":
        height = random.randint(170, 192)
        weight = random.randint(68, 85)

    else:
        height = random.randint(170, 190)
        weight = random.randint(65, 85)

    return f"{height} cm", f"{weight} kg"

<h2>Tách dữ liệu cầu thủ và thống kê của cầu thủ trong mùa giải</h2>
<p>Note: yellowred card là số lần nhận thẻ vàng thứ 2 , tức là trước đó đã nhận thẻ vàng rồi -> nhận thẻ đỏ gián tiếp</p>

In [20]:
import random
player_data = [] 
statistic_data = [] 
count = 0 
player_ids = set()
for raw_data in raw:
    player_inf = raw_data["data"]["response"]

    for item in player_inf:

        player = item["player"]

        # lấy position trước
        position = None
        if item["statistics"]:
            position = item["statistics"][0]["games"]["position"]

        height = player["height"]
        weight = player["weight"]

        # nếu null thì random
        if height is None or weight is None:
            random_height, random_weight = generate_height_weight(position)

            if height is None:
                height = random_height

            if weight is None:
                weight = random_weight


        if player["id"] not in player_ids:

            player_data.append({
                "id": player["id"],
                "name": player["name"],
                "firstname": player["firstname"],
                "lastname": player["lastname"],
                "date_birth": player["birth"]["date"],
                "nationality": player["nationality"],
                "height": height,
                "weight": weight,
                "image_url": player["photo"]
            })

            player_ids.add(player["id"])

        statistics = item["statistics"]

        check = False
        team_id = None
        position = None

        for i in statistics:

            team_id = i["team"]["id"]
            position = i["games"]["position"]

            if i["league"]["id"] != 39:
                continue

            check = True

            appearances = i["games"]["appearences"] or 0
            lineups = i["games"]["lineups"] or 0
            minutes = i["games"]["minutes"] or 0

            total_shots = i["shots"]["total"] or 0
            total_goals = i["goals"]["total"] or 0
            total_assists = i["goals"]["assists"] or 0
            total_saves = i["goals"]["saves"] or 0
            total_passes = i["passes"]["total"] or 0
            yellow_cards = i["cards"]["yellow"] or 0
            yellowred_cards = i["cards"]["yellowred"] or 0
            red_cards = i["cards"]["red"] or 0

            has_stats = any([
                total_shots,
                total_goals,
                total_assists,
                total_saves,
                total_passes,
                yellow_cards,
                yellowred_cards,
                red_cards
            ])

            if appearances == 0 and has_stats:
                appearances = random.randint(2, 15)
                lineups = random.randint(0, appearances)
                minutes = appearances * random.randint(20, 90)

            lineups = min(lineups, appearances)

            statistic_data.append({
                "player_id": player["id"],
                "team_id": team_id,
                "appearances": appearances,
                "position": position,
                "lineups": lineups,
                "minutes": minutes,
                "total_shots": total_shots,
                "total_goals": total_goals,
                "total_assists": total_assists,
                "total_saves": total_saves,
                "total_passes": total_passes,
                "yellow_cards": yellow_cards,
                "yellowred_cards": yellowred_cards,
                "red_cards": red_cards
            })

            break

        # player không có stat EPL
        if not check:

            statistic_data.append({
                "player_id": player["id"],
                "team_id": team_id,
                "appearances": 0,
                "position": position,
                "lineups": 0,
                "minutes": 0,
                "total_shots": 0,
                "total_goals": 0,
                "total_assists": 0,
                "total_saves": 0,
                "total_passes": 0,
                "yellow_cards": 0,
                "yellowred_cards": 0,
                "red_cards": 0
            })

In [22]:
print(len(player_data))
for item in player_data:
    if not item["weight"] or not item["height"]: print(item)
# lưu teams.json
with open("../database/players.json", "w", encoding="utf-8") as f:
    json.dump(player_data, f, ensure_ascii=False, indent=4)

290


<h2>Một player có thể có 2 thống kê thi đâu do có thể chơi cho 2 câu lạc bộ khác nhau trong một mùa</h2>

In [16]:
print(len(statistic_data))
for item in statistic_data:
    print(item)

300
{'player_id': 883, 'team_id': 33, 'appearances': 0, 'position': 'Goalkeeper', 'lineups': 0, 'minutes': 0, 'total_shots': 0, 'total_goals': 0, 'total_assists': 0, 'total_saves': 0, 'total_passes': 0, 'yellow_cards': 0, 'yellowred_cards': 0, 'red_cards': 0}
{'player_id': 889, 'team_id': 33, 'appearances': 20, 'position': 'Defender', 'lineups': 14, 'minutes': 1360, 'total_shots': 3, 'total_goals': 0, 'total_assists': 0, 'total_saves': 0, 'total_passes': 927, 'yellow_cards': 1, 'yellowred_cards': 0, 'red_cards': 0}
{'player_id': 895, 'team_id': 33, 'appearances': 2, 'position': 'Midfielder', 'lineups': 1, 'minutes': 88, 'total_shots': 6, 'total_goals': 0, 'total_assists': 1, 'total_saves': 0, 'total_passes': 298, 'yellow_cards': 0, 'yellowred_cards': 0, 'red_cards': 0}
{'player_id': 2931, 'team_id': 33, 'appearances': 0, 'position': 'Goalkeeper', 'lineups': 0, 'minutes': 0, 'total_shots': 0, 'total_goals': 0, 'total_assists': 0, 'total_saves': 0, 'total_passes': 0, 'yellow_cards': 0, '

In [17]:
with open("../database/players_statistic_season.json", "w", encoding="utf-8") as f:
    json.dump(statistic_data, f, ensure_ascii=False, indent=4)